[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/OWNER/REPO/blob/main/Lab_Notebooks/chapter_06_lab.ipynb)

*To run on Colab, replace `OWNER/REPO` in this badge link **and** in the `GITHUB_RAW` line of the setup cell with your GitHub path after pushing the repository. Everything else installs and loads automatically.*

# Chapter 6 — Model Selection and Regularisation
## Lab: subset selection, ridge, lasso, PCR, PLS

**Course:** An Introduction to Statistical Learning  
**Instructor:** Prof. Dr. Christoph Weisser, HSBI  
**Source:** James, Witten, Hastie, Tibshirani & Taylor (2023), *An Introduction to Statistical Learning, with Applications in Python*, Springer. Companion code at [statlearning.com](https://www.statlearning.com).


**Goal.** On the Hitters data, fit best-subset, ridge, lasso, PCR, and PLS, and compare them by cross-validation.


## Setup

Run this cell once. The `ISLP` package can be installed with `pip install ISLP`. As an alternative, the same data sets are available as CSVs in the workspace's `ALL CSV FILES - 2nd Edition` folder.


> **Google Colab:** this notebook also runs on Colab out of the box — the setup cell below installs any missing packages and (once the repo is on GitHub and `GITHUB_RAW` is set) downloads the data automatically.



In [ ]:
# --- Setup: runs locally AND on Google Colab --------------------------------
import importlib.util, os, subprocess, sys

IN_COLAB = 'google.colab' in sys.modules

def _ensure(pkg, import_name=None):
    """pip-install pkg (quietly) if its import is missing."""
    if importlib.util.find_spec(import_name or pkg) is None:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pkg], check=False)

if IN_COLAB:  # Colab ships numpy/pandas/sklearn/statsmodels; add course extras
    for _pkg, _imp in [('ISLP', 'ISLP')]:
        _ensure(_pkg, _imp)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

rng = np.random.default_rng(2024)
plt.rcParams['figure.dpi'] = 110

try:
    from ISLP import load_data
    HAVE_ISLP = True
except ImportError:
    HAVE_ISLP = False
    print('ISLP not installed; using CSV / URL fallbacks.')

# Local CSV location (repo layout first, then legacy paths, then a data/ cache).
_CANDIDATES = ['../ALL CSV FILES - 2nd Edition',
               'ALL CSV FILES - 2nd Edition',
               '../../ALL CSV FILES - 2nd Edition', 'data']
CSV = next((p for p in _CANDIDATES if os.path.isdir(p)), 'data')

# After pushing to GitHub, set OWNER/REPO so a fresh Colab runtime can fetch any
# CSV that is neither in ISLP nor already local (spaces in the folder -> %20).
GITHUB_RAW = ('https://raw.githubusercontent.com/OWNER/REPO/main/'
              'ALL%20CSV%20FILES%20-%202nd%20Edition')

# The three datasets NOT in the ISLP package -> load from the book's official
# site so the notebook works on a fresh Colab even before the repo is published.
KNOWN_URLS = {
    'Advertising': 'https://www.statlearning.com/s/Advertising.csv',
    'Heart':       'https://www.statlearning.com/s/Heart.csv',
    'Income1':     'https://www.statlearning.com/s/Income1.csv',
    'Income2':     'https://www.statlearning.com/s/Income2.csv',
}

def load(name, **read_csv_kwargs):
    """Load a course dataset. Order: ISLP package -> R datasets -> local CSV
    -> official book URL -> your GitHub repo. Works locally and on Colab."""
    if HAVE_ISLP:
        try:
            return load_data(name)
        except Exception:
            pass
    if name == 'USArrests':                       # classic R dataset, not in ISLP
        try:
            import statsmodels.api as sm
            return sm.datasets.get_rdataset('USArrests', 'datasets').data
        except Exception:
            pass
    path = f'{CSV}/{name}.csv'
    if os.path.exists(path):                      # running from the repo (local)
        return pd.read_csv(path, **read_csv_kwargs)
    remotes = ([KNOWN_URLS[name]] if name in KNOWN_URLS else []) + [f'{GITHUB_RAW}/{name}.csv']
    for url in remotes:                           # fresh Colab: stream over https
        try:
            return pd.read_csv(url, **read_csv_kwargs)
        except Exception:
            continue
    raise FileNotFoundError(
        f"Could not load {name!r}. Put the CSV in '{CSV}/' or set OWNER/REPO in GITHUB_RAW.")

## 1. Hitters data


In [ ]:
Hitters = load('Hitters').dropna().reset_index(drop=True)
y = Hitters['Salary']
X = Hitters.drop(columns='Salary')
X = pd.get_dummies(X, drop_first=True).astype(float)
print(X.shape); X.head()


## 2. Best-subset and forward stepwise selection
Brute-force best-subset for small $k$; forward stepwise for comparability.


In [ ]:
import itertools, statsmodels.api as sm
from sklearn.metrics import mean_squared_error
preds = X.columns.tolist()
best = {}
for k in range(1, 6):
    best_rss, best_combo = None, None
    for combo in itertools.combinations(preds, k):
        rss = sm.OLS(y, sm.add_constant(X[list(combo)])).fit().ssr
        if best_rss is None or rss < best_rss:
            best_rss, best_combo = rss, combo
    best[k] = best_combo
best


In [ ]:
# Forward stepwise
remaining = preds.copy(); selected = []
for step in range(8):
    best_rss, best_v = None, None
    for v in remaining:
        rss = sm.OLS(y, sm.add_constant(X[selected + [v]])).fit().ssr
        if best_rss is None or rss < best_rss:
            best_rss, best_v = rss, v
    selected.append(best_v); remaining.remove(best_v)
    print(step + 1, '->', selected[-1], 'RSS =', round(best_rss, 1))


## 3. Ridge regression


In [ ]:
from sklearn.linear_model import RidgeCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
alphas = np.logspace(-2, 4, 50)
ridge = make_pipeline(StandardScaler(),
                       RidgeCV(alphas=alphas)).fit(X, y)
print('best alpha:', ridge.named_steps['ridgecv'].alpha_)
print(pd.Series(ridge.named_steps['ridgecv'].coef_, index=X.columns).round(2))


### Ridge coefficient paths


In [ ]:
from sklearn.linear_model import Ridge
Xs = StandardScaler().fit_transform(X)
paths = np.array([Ridge(alpha=a).fit(Xs, y).coef_ for a in alphas])
fig, ax = plt.subplots(figsize=(7, 4))
for j, name in enumerate(X.columns):
    ax.plot(alphas, paths[:, j], label=name)
ax.set_xscale('log'); ax.set_xlabel('alpha'); ax.set_ylabel('coef')
ax.set_title('Ridge paths'); plt.show()


## 4. Lasso


In [ ]:
from sklearn.linear_model import LassoCV, Lasso
alphas = np.logspace(-3, 2, 50)
lasso = make_pipeline(StandardScaler(),
                       LassoCV(alphas=alphas, cv=10, random_state=0)).fit(X, y)
print('best alpha:', lasso.named_steps['lassocv'].alpha_)
coefs = pd.Series(lasso.named_steps['lassocv'].coef_, index=X.columns)
print('non-zero:', coefs[coefs != 0].round(2))


### Lasso paths


In [ ]:
Xs = StandardScaler().fit_transform(X)
paths = np.array([Lasso(alpha=a, max_iter=10000).fit(Xs, y).coef_ for a in alphas])
fig, ax = plt.subplots(figsize=(7, 4))
for j in range(paths.shape[1]):
    ax.plot(alphas, paths[:, j])
ax.set_xscale('log'); ax.set_xlabel('alpha'); ax.set_ylabel('coef')
ax.set_title('Lasso paths'); plt.show()


## 5. PCR and PLS


In [ ]:
from sklearn.decomposition import PCA
from sklearn.cross_decomposition import PLSRegression
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import cross_val_score
Xs = StandardScaler().fit_transform(X)
pcr_scores = []; pls_scores = []
for M in range(1, X.shape[1] + 1):
    pcr = make_pipeline(PCA(n_components=M), LinearRegression())
    pls = PLSRegression(n_components=M, scale=False)
    pcr_scores.append(-cross_val_score(pcr, Xs, y, cv=10,
                                         scoring='neg_mean_squared_error').mean())
    pls_scores.append(-cross_val_score(pls, Xs, y, cv=10,
                                         scoring='neg_mean_squared_error').mean())
fig, ax = plt.subplots(figsize=(6, 4))
Ms = range(1, X.shape[1] + 1)
ax.plot(Ms, pcr_scores, marker='o', label='PCR')
ax.plot(Ms, pls_scores, marker='s', label='PLS')
ax.set(xlabel='M', ylabel='10-fold CV MSE'); ax.legend(); plt.show()


## Lecture exercises — worked Python solutions

The cells below mirror the **[Python]-tagged exercises** from the Chapter 6 lecture slides, with fully worked, runnable solutions and the expected numeric answers (stored as inline comments). Each solution reloads `Hitters` through the `load()` helper so it stands alone and reproduces the slide numbers.

### Exercise 6.7 — CV ridge and lasso on Hitters

On the `Hitters` data (predict `Salary`): (i) fit the lasso with a cross-validated $\lambda$ on standardised predictors, (ii) report the selected $\lambda$, and (iii) list which features the lasso keeps versus drops. Contrast with ridge, which keeps every predictor.

In [ ]:
# Exercise 6.7 -- lasso vs ridge with a cross-validated penalty on Hitters.
from sklearn.linear_model  import LassoCV, RidgeCV
from sklearn.preprocessing import StandardScaler

df    = load('Hitters').dropna()                 # drop the 59 rows with NA Salary
yH    = df['Salary'].values
X_raw = pd.get_dummies(df.drop(columns='Salary'),    # encode League/Division/NewLeague
                       drop_first=True).astype(float)
preds = X_raw.columns
XH    = StandardScaler().fit_transform(X_raw)    # standardise: penalty is scale-dependent
print('n, p =', XH.shape)                        # (263, 19)

lasso = LassoCV(cv=10, random_state=0, max_iter=10000).fit(XH, yH)  # CV lambda
print('lasso alpha :', round(lasso.alpha_, 3))   # 2.553
kept    = [p for p, c in zip(preds, lasso.coef_) if abs(c) >  1e-8]
dropped = [p for p, c in zip(preds, lasso.coef_) if abs(c) <= 1e-8]
print('kept    :', kept)      # Hits, Walks, CRBI, PutOuts, Division_W, ...
print('dropped :', dropped)   # HmRun, Runs, RBI, CAtBat, CHits, NewLeague_N

ridge = RidgeCV(alphas=np.logspace(-2, 4, 50)).fit(XH, yH)   # CV lambda for ridge
print('ridge alpha :', round(ridge.alpha_, 3),
      '-- nonzero coefs:', int(np.sum(np.abs(ridge.coef_) > 1e-8)))   # 3.728 -- 19

**Interpretation.** The lasso selects a moderate $\lambda\approx2.55$ and drives several coefficients *exactly* to zero — it keeps strong career/recent-performance predictors (e.g. `Hits`, `Walks`, `CRBI`, `PutOuts`, `Division_W`) and drops weak or redundant ones (`HmRun`, `Runs`, `RBI`, `CAtBat`, `CHits`, `NewLeague_N`). Ridge keeps all 19 predictors nonzero (it only shrinks). Takeaway: the lasso yields a sparse, more interpretable model via automatic variable selection, whereas ridge stabilises the fit while retaining every predictor.

### Extended Exercise 6.2 — CV ridge and lasso *test* MSE on Hitters

Split `Hitters` into train/test, standardise using the **training set only** (no leakage), and cross-validate $\lambda$ for both ridge and lasso. Report each chosen $\lambda$, the *test* MSE, and how many coefficients are non-zero; list the predictors the lasso drops.

In [ ]:
# Extended Exercise 6.2 -- honest test-MSE comparison with leak-free scaling.
from sklearn.model_selection import train_test_split
from sklearn.preprocessing   import StandardScaler
from sklearn.linear_model    import RidgeCV, LassoCV
from sklearn.metrics         import mean_squared_error

df  = load('Hitters').dropna(subset=['Salary'])  # drop rows with NA response
yH  = df['Salary'].values
XH  = pd.get_dummies(df.drop(columns='Salary'),
                     drop_first=True).astype(float)
Xtr, Xte, ytr, yte = train_test_split(XH.values, yH,
                        test_size=0.33, random_state=1)    # 2/3 - 1/3 split
sc = StandardScaler().fit(Xtr)                   # fit scaler on TRAIN only
Xtr, Xte = sc.transform(Xtr), sc.transform(Xte)  # apply to both -> no leakage

ridge = RidgeCV(alphas=np.logspace(-2, 4, 50)).fit(Xtr, ytr)
lasso = LassoCV(cv=10, max_iter=100000, random_state=0).fit(Xtr, ytr)
for name, m in [('ridge', ridge), ('lasso', lasso)]:
    mse = mean_squared_error(yte, m.predict(Xte))        # TEST-set MSE
    nz  = int(np.sum(np.abs(m.coef_) > 1e-8))            # nonzero coefficients
    print(f'{name}: lambda={round(m.alpha_, 3)}  testMSE={round(mse, 0)}  nonzero={nz}')
    # ridge: lambda=0.095  testMSE=122695.0  nonzero=19
    # lasso: lambda=0.779  testMSE=119064.0  nonzero=16
dropped = [c for c, w in zip(XH.columns, lasso.coef_) if abs(w) <= 1e-8]
print('lasso drops:', dropped)                   # ['Years', 'CRBI', 'Errors']

**Interpretation.** With $n=263$, $p=19$ and this split, ridge chooses $\lambda\approx0.1$ (test MSE $\approx1.2\times10^5$, all 19 coefficients non-zero) while the lasso chooses $\lambda\approx0.8$ (test MSE $\approx1.2\times10^5$, 16 non-zero) and drops `Years`, `CRBI`, `Errors` (weak or redundant given the correlated career totals). Both attain nearly identical test error, but the lasso reaches it with a *sparser, more interpretable* model. Exact numbers shift with the split/seed; the qualitative message — lasso selects, ridge keeps all — is stable.

## 6. Exercises
1. Run *backward* stepwise selection and compare with forward.
2. Use the one-standard-error rule to choose a more parsimonious lasso $\lambda$.
3. Fit elastic net on Hitters with `ElasticNetCV`. How does it compare to ridge and lasso?
4. On a high-dimensional problem ($p > n$), demonstrate that OLS fails while ridge / lasso work.
